# IMPORT

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from pandas_profiling import ProfileReport
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import seaborn as sns
from preprocess import preprocess
import plotly.express as px
import shutil
from sklearn_pandas import DataFrameMapper
import sqlite3 as sq
import os
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/profiles_revised.csv')
df.columns = df.columns.str.replace('\t', '') # needed?
origin = df.columns

In [ ]:
if os.path.exists('./exploration/'):
    shutil.rmtree('./exploration/', ignore_errors=True)

In [ ]:
def print_col_values(list, filename):
    with open(r'{}.txt'.format(filename), 'w') as fp:
        for element in list:
            fp.write("{}\n".format(element))

In [ ]:
def df_distinct_values(df, folder):
    if not os.path.exists(folder):
        os.makedirs('./exploration/'+folder)
    for (index, colname) in enumerate(df.columns):
        #print(index, colname)
        distinc_values = df[colname].unique()
        print_col_values(list=distinc_values, filename='./exploration/{}/{}-{}-values'.format(folder, index, colname))

In [ ]:
df_distinct_values(df=df, folder='origin')

In [ ]:
profile = ProfileReport(df, title='Pandas Profilign Report')
profile.to_notebook_iframe()
profile.to_file('./exploration/pandas_profiling_data_report.html')

# CLEAN

In [ ]:
#df = df[['age', 'body_type']]
df_clean = preprocess(df.columns, df)

In [ ]:
speaks_cols = [
    'speaks_english', 'speaks_spanish', 'speaks_french', 'speaks_c++',
    'speaks_chinese', 'speaks_tagalog', 'speaks_portuguese',
    'speaks_japanese', 'speaks_russian', 'speaks_ukrainian',
    'speaks_sanskrit', 'speaks_thai', 'speaks_hindi', 'speaks_sign',
    'speaks_swedish', 'speaks_german', 'speaks_italian', 'speaks_arabic',
    'speaks_latin', 'speaks_other', 'speaks_hebrew', 'speaks_hawaiian',
    'speaks_korean', 'speaks_ancient', 'speaks_vietnamese',
    'speaks_indonesian', 'speaks_latvian', 'speaks_hungarian',
    'speaks_lisp', 'speaks_swahili', 'speaks_rotuman', 'speaks_czech',
    'speaks_yiddish', 'speaks_greek', 'speaks_catalan', 'speaks_croatian',
    'speaks_farsi', 'speaks_icelandic', 'speaks_tamil', 'speaks_serbian',
    'speaks_esperanto', 'speaks_norwegian', 'speaks_bengali',
    'speaks_dutch', 'speaks_urdu', 'speaks_irish', 'speaks_welsh',
    'speaks_sign_language', 'speaks_khmer', 'speaks_cebuano',
    'speaks_afrikaans', 'speaks_albanian', 'speaks_romanian',
    'speaks_polish', 'speaks_turkish', 'speaks_finnish']

In [ ]:
speaks = []
for speaks_col in speaks_cols:
    speaks.append((speaks_col, df_clean[speaks_col].sum()))
print(speaks)

In [ ]:
sorted_speaks = sorted(speaks, key=lambda x: x[1] , reverse=True)
sorted_speaks

In [ ]:
speaks_cols = [
    'speaks_tagalog', 'speaks_portuguese',
    'speaks_russian', 'speaks_ukrainian',
    'speaks_sanskrit', 'speaks_thai', 'speaks_hindi', 'speaks_sign',
    'speaks_swedish', 'speaks_arabic',
    'speaks_latin', 'speaks_other', 'speaks_hebrew', 'speaks_hawaiian',
    'speaks_korean', 'speaks_ancient', 'speaks_vietnamese',
    'speaks_indonesian', 'speaks_latvian', 'speaks_hungarian',
    'speaks_lisp', 'speaks_swahili', 'speaks_rotuman', 'speaks_czech',
    'speaks_yiddish', 'speaks_greek', 'speaks_catalan', 'speaks_croatian',
    'speaks_farsi', 'speaks_icelandic', 'speaks_tamil', 'speaks_serbian',
    'speaks_esperanto', 'speaks_norwegian', 'speaks_bengali',
    'speaks_dutch', 'speaks_urdu', 'speaks_irish', 'speaks_welsh',
    'speaks_sign_language', 'speaks_khmer', 'speaks_cebuano',
    'speaks_afrikaans', 'speaks_albanian', 'speaks_romanian',
    'speaks_polish', 'speaks_turkish', 'speaks_finnish']

In [ ]:
df_clean = df_clean.drop(columns=speaks_cols)

In [ ]:
df_clean

In [ ]:
# TODO: fix '\'
#df_distinct_values(df=df_clean, folder='cleaned')

# SAVE

In [ ]:
df_clean.to_csv('./data/cleaned.csv')

# STANDARDIZE

In [ ]:
df_clean.head()
sample = df_clean.iloc[:1]

In [ ]:
continuous_cols = ['age', 'height']
categorical_cols = ['body_type', 'drinks', 'drugs', 'income', 'job', 'orientation', 'sex', 'smokes', 'status',
'diet','diet_modifier',
'education_status', 'education_institution',
'offspring_status', 'offspring_future',
'pets_cats', 'pets_dogs',
'religion_type', 'religion_modifier',
'sign', 'sign_modifier']
ethnities_cols = df_clean[df_clean.columns[pd.Series(df_clean.columns).str.startswith('ethnicities')]].columns
speaks_cols = df_clean[df_clean.columns[pd.Series(df_clean.columns).str.startswith('speaks')]].columns

In [ ]:
# Example
mapper = DataFrameMapper([
  ('body_type', LabelEncoder()),
  (['age'], StandardScaler())],
  #[(categorical_col, LabelBinarizer()) for categorical_col in categorical_cols],
  df_out=True 
)
print(mapper)

In [ ]:
# Mapper for checking
mapper = DataFrameMapper([
  #('drinks', LabelEncoder()),
  (['age'], StandardScaler())] +
  [(categorical_col, LabelBinarizer()) for categorical_col in categorical_cols],
  df_out=True 
)
print(mapper)

In [ ]:
# Real mapper
mapper = DataFrameMapper(
  [([continuous_col], StandardScaler()) for continuous_col in continuous_cols] +
  [(categorical_col, LabelEncoder()) for categorical_col in categorical_cols] +
  [(ethnities_col, LabelEncoder()) for ethnities_col in ethnities_cols] +
  [(speaks_col, LabelEncoder()) for speaks_col in speaks_cols],
  df_out=True 
)
mapper

In [ ]:
df_std = np.round(mapper.fit_transform(df_clean.copy()),2)

In [ ]:
table_names = ['okcupid_clean', 'okcupid_std']
#table_path = './data/'+ table_name +'_db'

dfs = {
    "std_clean": df_clean,
    "dt_std": df_std,
}

with sq.connect('okcupid.sqlite') as db:
    df_clean.to_sql('okcupid_clean', db, if_exists='replace', index=True)
    df_std.to_sql('okcupid_std', db, if_exists='replace', index=True)


In [ ]:
sample

In [ ]:
np.round(mapper.transform(sample), 2)

In [ ]:
# https://scikit-learn.org/stable/modules/preprocessing.html
# https://stackoverflow.com/questions/43554821/feature-preprocessing-of-both-continuous-and-categorical-variables-of-integer-t
# https://stackoverflow.com/questions/53152627/saving-standardscaler-model-for-use-on-new-datasets?noredirect=1&lq=1
# https://stackoverflow.com/questions/38780302/predicting-new-data-using-sklearn-after-standardizing-the-training-data

# Exploration

In [ ]:
plt.figure(figsize=(5,5))
sns.heatmap(df.corr(), center=0, annot=True)
plt.title('Correlation Map')
plt.show()
plt.savefig('./exploration/correlation-map.png')

In [ ]:
#plt.figure(figsize=(5,5))
#sns.pairplot(data=df, hue='lables', palette='RdBu')
#plt.title('Correlation Map')
#plt.show()
#plt.savefig('./exploration/pairplot.pdf')

In [ ]:
if not os.path.exists('./exploration/features'):
        os.makedirs('./exploration/features')
# TAKES FOREVER
for n_index, column in enumerate(df_std.columns):
    for m_index, column_iterator in enumerate(df_std.columns):
        pass
        print(n_index, column, m_index, column_iterator)
        plt.figure(figsize=(5, 5))
        plt.scatter(df_std.iloc[:, n_index], df_std.iloc[:, m_index])
        plt.xlabel(column)
        plt.ylabel(column_iterator)
        plt.title('Feature {} vs. Feature {}'.format(column, column_iterator))
        filename = '{}-{}-scatter.pdf'.format(column, column_iterator)
        filename = filename.replace('/', '-')
        plt.savefig('./exploration/features/{}'.format(filename))

In [ ]:
# TAKES FOREVER
pd.plotting.scatter_matrix(df_std, alpha=0.2)
plt.savefig('./exploration/correlation-map.pdf')

# PCA

In [ ]:
pca = PCA()
pca.fit(df_std)
var_ratio = pca.explained_variance_ratio_

In [ ]:
fig = plt.figure(figsize=(10,8))
plt.plot(range(1, len(var_ratio)+1), var_ratio.cumsum(), marker='o', linestyle='--')
plt.title('Explaind Variance by Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
fig.savefig('./exploration/variance-by-principal-components.pdf')

In [ ]:
PCA_COMPONENTS = 4

In [ ]:
pca = PCA(n_components=PCA_COMPONENTS)
pca.fit(df_std)
scores_pca = pca.transform(df_std)
print(scores_pca)

In [ ]:
wcss = []
for i in range(1, 21):
    kmeans_pca = KMeans(n_clusters=i, init='k-means++', random_state=420)
    kmeans_pca.fit(scores_pca)
    wcss.append(kmeans_pca.inertia_)

In [ ]:
fig = plt.figure(figsize = (10,8))
plt.plot(range(1, 21), wcss, marker='o', linestyle='--')
plt.title('Number of Clusters')
plt.xlabel('WWCSS')
plt.ylabel('K-means witch PCA Clustering')
fig.savefig('./exploration/number-of-clusters.pdf')

In [ ]:
OPTIMAL_N_CLUSTER = 4

In [ ]:
kmeans_pca = KMeans(n_clusters=OPTIMAL_N_CLUSTER, init='k-means++', random_state=420)
kmeans_pca.fit(scores_pca)

In [ ]:
df_segm_pca_kmeans = pd.concat([df_std.reset_index(drop=True), pd.DataFrame(scores_pca)], axis=1)
df_segm_pca_kmeans.columns.values[-PCA_COMPONENTS:] = ['PComp 1', 'PComp 2', 'PComp 3', 'PComp 4']

df_segm_pca_kmeans['Segment K-means PCA'] = kmeans_pca.labels_

In [ ]:
df_segm_pca_kmeans

In [ ]:
df_segm_pca_kmeans['Segment'] = df_segm_pca_kmeans['Segment K-means PCA'].map({0: 'first',
    1: 'second',
    2: 'third',
    3: 'fourth'
})

In [ ]:
df_segm_pca_kmeans.head()

In [ ]:
x_axis = df_segm_pca_kmeans['PComp 1']
y_axis = df_segm_pca_kmeans['PComp 2']
plt.figure(figsize=(10,8))
sns.scatterplot(x=x_axis, y=y_axis, hue = df_segm_pca_kmeans['Segment'], palette=['g', 'r', 'c', 'm'])
plt.show()

In [ ]:
df_cluster = df_segm_pca_kmeans.iloc[:,-6:] # 4 components + hue + cat
df_cluster.head

In [ ]:
if not os.path.exists('./exploration/PCA'):
        os.makedirs('./exploration/PCA')

for n_index, column in enumerate(df_cluster.columns[:PCA_COMPONENTS]):
    for m_index, column_iterator in enumerate(df_cluster.columns[:PCA_COMPONENTS]):
        #print(n_index, column, m_index, column_iterator)
        fig = plt.figure(figsize=(5, 5))
        sns.scatterplot(x=df_cluster[column], y=df_cluster[column_iterator], hue = df_segm_pca_kmeans['Segment'], palette=['g', 'r', 'c', 'm'])
        plt.xlabel(column)
        plt.ylabel(column_iterator)
        plt.title('Clusters by PCA Component')
        fig.savefig('./exploration/PCA/PCA {} vs {}.pdf'.format(column, column_iterator)) 
        #plt.show()

In [ ]:
fig = px.scatter(df_cluster, x="PComp 1", y="PComp 2", color='Segment', text=df_cluster.index)
fig.show()